In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas
from musicdb import PanDBIO
from gate import IOStore
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
hio = HTMLIO()
io  = FileIO()
mp     = MasterParams(verbose=True)
dbs    = mp.getDBs()
pdbio  = PanDBIO()
ios    = IOStore()

# Classical Archives

In [ ]:
mdbio = ios.get("Spotify", verbose=True)
for modVal in range(1,100):
    mdbio.prd.parseAlbumData(modVal, force=True)

In [ ]:
#mdbio.prd.mergeModValData()


In [ ]:
data['3eDtd9oQyjD2OuMNgVHOag'].media.media['album + album'][0].get()

In [ ]:
if True:
    from utils import PoolIO
    pio = PoolIO("Spotify")
    #pio.parse(force=True)
    pio.meta()
    pio.sum()
    pio.search()

In [ ]:
data = io.get("/Users/tgadfort/Music/Discog/artists-spotify/mv-00-gv-00.p")

In [ ]:

from lib.genius import MusicDBIO
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=False)
mdbioLocal.dir.getRawSongModValDataDir(0).path

In [ ]:
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
for modVal in range(100):
    mdbioGlobal.dir.getRawAlbumModValDataDir(modVal).mkDir()
    mdbioGlobal.dir.getRawMasterModValDataDir(modVal).mkDir()

# Create Songs Raw Data

In [ ]:
libdb = "allmusic"

def getModGlobVal(modVal):
    retval = f"0{modVal}" if modVal < 10 else f"{modVal}"
    return retval

def getGlobVal(mdbioGlobal, basename):
    if hasattr(mdbioGlobal, "getGlobVal") and callable(getattr((mdbioGlobal, "getGlobVal"))):
        modVal  = mdbioGlobal.getModVal(basename)
        globVal = mdbioGlobal.getGlobVal(basename)
    else:
        modVal  = getModGlobVal(int(basename[-2]))
        globVal = getModGlobVal(int(basename[-4:-2]))
    return globVal

## Merge global data

In [ ]:
exec(f"from lib.{libdb} import MusicDBIO")
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=False)
db          = mdbioGlobal.db
dType       = {"Genius": "Song", "Discogs": "Master", "Deezer": "Album", "AllMusic": "Artist", "Spotify": "Album"}[db]
dstDir      = DirInfo(mdbioLocal.dir.getRawDataDir())
prevDir     = DirInfo(mdbioLocal.dir.getRawDataDir().join("prev"))
prevDir.mkDir()

ts = Timestat("Creating ModVal Files")
modVals  = {mVal: getModGlobVal(mVal) for mVal in range(1)}
globVals = {gVal: getModGlobVal(gVal) for gVal in range(100)}

for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    srcDir  = eval(f"mdbioLocal.dir.getRaw{dType}ModValDataDir(modVal)")
    if db == "AllMusic" and dType == "Artist":
        srcDir  = eval(f"mdbioLocal.dir.getRawModValDataDir(modVal)")
    elif db == "Spotify" and dType == "Album":
        srcDir  = eval(f"mdbioLocal.dir.getRawModValDataDir(modVal)")
        
    gfiles = [FileInfo(ifile) for ifile in srcDir.glob("*.p", debug=False)]
    gfiles = {finfo: getGlobVal(mdbioGlobal, finfo.basename) for finfo in gfiles}    
    if len(gfiles) == 0:
        print("No files...")
        continue
        
    for g,(gVal,globVal) in enumerate(globVals.items()):
        if g % 25 == 0: print(f"ModVal={modGlobVal}   GlobVal={globVal}   ", end="")

        files = [finfo for finfo,finfoGlobVal in gfiles.items() if finfoGlobVal == globVal]
        if len(files) == 0:
            if g % 25 == 0: print("Data=0")
            continue
            
        savename = dstDir.join(f"{db}-{dType}-mv-{modGlobVal}-gv-{globVal}.p")            
        retval = {}
        for finfo in files:
            try:
                retval[finfo.basename] = io.get(finfo)
            except:
                print(f"ERROR With {finfo.str}")
        
        if g % 25 == 0: print(f"Data={len(retval)}    ", end="")
        if g % 25 == 0: print(f"Saved={savename.str}")
        io.save(idata=retval, ifile=savename)

        for finfo in files:
            dstFile = prevDir.join(finfo.name)
            finfo.mvFile(dstFile, debug=False)

    ts.update(n=n+1,N=len(modVals))
ts.stop()

## Rename mv-gv to official style

In [ ]:
exec(f"from lib.{libdb} import MusicDBIO")
mdbioGlobal = MusicDBIO(local=True, mkDirs=False)
db = mdbioGlobal.db
dType = "Album"

modVals  = {mVal: getModGlobVal(mVal) for mVal in range(1)}
globVals = {gVal: getModGlobVal(gVal) for gVal in range(100)}
for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    break
    #dstDir  = eval(f"mdbioGlobal.dir.getRaw{dType}ModValDataDir(modVal)")
    srcDir  = eval(f"mdbioGlobal.dir.getRawDataDir()")
    dstDir  = srcDir
    for g,(gVal,globVal) in enumerate(globVals.items()):
        srcFile = srcDir.join(f"{db}-Artist-mv-{modGlobVal}-gv-{globVal}.p")
        #srcFile = srcDir.join(f"mv-{modGlobVal}-gv-{globVal}.p")
        #print(srcFile.path)
        #break
        if srcFile.exists():
            dstFile = dstDir.join(f"{db}-{dType}-mv-{modGlobVal}-gv-{globVal}.p")
            #print(srcFile.path)
            #print(dstFile.path)
            #1/0
            srcFile.mvFile(dstFile)

## Merge or move local files to global

In [ ]:
exec(f"from lib.{libdb} import MusicDBIO")
from fileutils import FileInfo, DirInfo
mdbioGlobal = MusicDBIO(local=False, mkDirs=False)
mdbioLocal  = MusicDBIO(local=True, mkDirs=False)
prevDir     = DirInfo(mdbioLocal.dir.getRawDataDir().join("prev"))
prevDir.mkDir()
db          = mdbioGlobal.db
dType       = {"Genius": "Song", "Discogs": "Master", "Deezer": "Album", "AllMusic": "Artist", "Spotify": "Album"}[db]
srcDir      = DirInfo(mdbioLocal.dir.getRawDataDir())

test = False
modVals  = {mVal: getModGlobVal(mVal) for mVal in range(1)}
for n,(modVal,modGlobVal) in enumerate(modVals.items()):
    dstDir  = eval(f"mdbioGlobal.dir.getRaw{dType}ModValDataDir(modVal)")
    globCmd = srcDir.glob(f"{db}-{dType}-mv-{modGlobVal}-gv-*.p")
    files = [FileInfo(ifile) for ifile in globCmd]
    if test is True:
        print(len(files))
    for g,finfo in enumerate(files):
        if g % 25 == 0: print(f"File={finfo.name}   ", end="")
        dstFile = dstDir.join(finfo.name)
        if test is True:
            print(finfo.path)
            print(dstFile.path)
            print(dstFile.exists())
            
        if dstFile.exists():
            localData = io.get(finfo)
            if g % 25 == 0: print(f"Local={len(localData)}    ", end="")
            globalData = io.get(dstFile)
            globalData = {} if globalData is None else globalData
            if g % 25 == 0: print(f"Global={len(globalData)}    ", end="")
            globalData.update(localData)
            if g % 25 == 0: print(f"Total={len(globalData)}    ", end="")
            if test is True:
                1/0
            io.save(idata=globalData, ifile=dstFile)
            prevFile = prevDir.join(finfo.name)
            finfo.mvFile(prevFile, debug=False)
            if g % 25 == 0: print(f"File={dstFile.str}")
        else:
            if test is True:
                1/0
            finfo.mvFile(dstFile)
            if g % 25 == 0: print(f"File={dstFile.str}")
    if test is True:
        break

In [ ]:
len(io.get("/Volumes/Piggy/Discog/artists-allmusic/0/artists/AllMusic-Artist-mv-00-gv-41.p"))

In [ ]:
files = modValDir.glob("*.p")

In [ ]:
for ifile in files:
    break

In [ ]:
data = io.get(ifile)

In [ ]:
ifile

In [ ]:
data

In [ ]:
def getTabsData(rData):
    tabs = {tab.title: tab.href for tab in rData.profile.extra.get('tabs', [])}
    name = rData.artist.name
    url  = rData.url.url
    retval = {"Name": name, "URL": url, **tabs}
    return retval

from timeutils import Timestat
ts = Timestat("Getting Tabs Data")
tabsData = modValData.apply(getTabsData).apply(Series)
tabsData["Tabs"] = tabsData.drop(['Name','URL'], axis=1).count(axis=1)
tabsData = tabsData.sort_values(by="Tabs", ascending=False)
ts.stop()

In [ ]:
tabsData[tabsData["Overview"].notna()].loc['0003218600']['URL']

In [ ]:
tabsData.shape

In [ ]:
ifile = "/Volumes/Piggy/Discog/artists-beatport/0/727300.p"
from lib.beatport import RawDBData
rdb = RawDBData()
retval = rdb.getArtistData(ifile)

In [ ]:
from lib.classicalarchives import moveLocalFiles
moveLocalFiles()
#mdbio.data.getR

In [ ]:
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
bsdata = hio.get(io.get('/Volumes/Piggy/Discog/artists-classicalarchives/0/70800.p'))

In [ ]:
modValData = mdbio.data.getModValComposerData(0)
#mdbio.prd.parseComposerData(0, force=True)

In [ ]:
tmp = io.get('/Volumes/Seagate/Discog/artists-classicalarchives-db/composer/0-DB.p')
tmp['70800'].show()

In [ ]:
from lib.classicalarchives import RawDBData
rdb = RawDBData()
retval = rdb.getComposerData("/Volumes/Piggy/Discog/artists-classicalarchives/0/70800.p")
                              /Volumes/Piggy/Discog/artists-classicalarchives/0/composer/70800.p

In [ ]:
mdb

In [ ]:
#modValData['70800'].show()
from lib.classicalarchives import moveLocalFiles, MusicDBIO
#moveLocalFiles()
mdbio = ios.get("ClassicalArchives", verbose=True)
#mdbio = MusicDBIO(verbose=True)

In [ ]:
from lib.classicalarchives import MusicDBIO
mdbio = MusicDBIO()
for modVal in range(100):
    mdbio.prd.parseComposerData(modVal)
    mdbio.prd.parsePerformerData(modVal)
mdbio.prd.mergeModValData()
mdbio.meta.make()
mdbio.sum.make()
mdbio.search.make()

In [ ]:
from musicdb import PanDBIO
pdbio = PanDBIO()
pdbio.addMetrics()
pdbio.setIndex()

In [ ]:

for modVal in range(100):
    mdbio.dir.getRawComposerModValDataDir(modVal).mkDir()
    mdbio.dir.getRawPerformerModValDataDir(modVal).mkDir()

In [ ]:
mdbio = ios.get("ClassicalArchives")
mdbio.data.getSummaryNameData()

In [ ]:
retval.show()

In [ ]:
ifile="/Users/tgadfort/Music/Discog/artists-beatport/16/track/574116.p"
from lib.beatport import RawDBData
from ioutils import FileIO, HTMLIO
bsdata = hio.get(io.get(ifile))

In [ ]:
rdb = RawDBData()
retval = rdb.getArtistData(ifile)

In [ ]:
retval.media.media['Tracks'][0].get()

In [ ]:
from bs4 import element
import json
scriptData = bsdata.find("script", {"id": "data-objects"})
jData      = {}
if isinstance(scriptData, element.Tag):
    try:
        lines = [line.strip().split(" = ") for line in scriptData.text.split("\n") if len(line) > 0]
        windowData = {line[0]: line[1] for line in lines if len(line) == 2}
        playables  = windowData["window.Playables"]
        jData      = json.loads(playables[:-1])
    except:
        jData      = {}

In [ ]:
len(jData['tracks'])

In [ ]:
data-objects

In [ ]:
modValData = mdbio.data.getModValData(0)

In [ ]:
ifile = "/Volumes/Piggy/Discog/artists-spotify/0/2Waw2sSbqvAwK8NwACNjVo.p"
from lib.spotify import RawDBData
rdb = RawDBData()
retval = rdb.getAlbumData(ifile)
retval.media.media['album + album'][0].get()

In [ ]:
mdbio.data.getRawArtistAlbumFilename("2Waw2sSbqvAwK8NwACNjVo")

In [ ]:
media = modValData['2Waw2sSbqvAwK8NwACNjVo'].media.media
for k,v in media.items():
    for v2 in v:
        print(v2.artist)

In [ ]:
dates = mdbio.data.getMetaDatesData(0)
dates

In [ ]:
#mdbio.prd.parse(0, force=True)
mdbio.meta.make(modVal=0)

In [ ]:
modValData['316200'].profile.get()

In [ ]:
mdbio.data.getMetaMetricData(0).max()

In [ ]:
mediaFormats = utils.getMediaFormats(modValData['316200'])
mediaGenres = [record.get('Genres', {}) for mediaType,mediaFormatData in mediaFormats.items() for record in mediaFormatData]
#retval = Series([record.text for item in mediaGenres for record in item if record is not None], dtype='object')
#return retval



In [ ]:
from listUtils import getFlatList


In [ ]:
mediaArtists = [record.artist for mediaType,mediaTypeData in media.items() for record in mediaTypeData]
retval = {artistID: artistName for record in mediaArtists for artistID,artistName in record.items()}

In [ ]:
retval

In [ ]:
from pandas import to_datetime
for k,v in modValData['316200'].media.media.items():
    print(f"==== {k} ====")
    for record in v:
        print(record.get())
        aformat = record.aformat
        print(aformat)
        date = aformat['Date']
        year = to_datetime(date).year
        print(f"\t{year}")

In [ ]:
db = "SetListFM"
dbids         = mmeDF[mmeDF[db].notna()][[db]].copy(deep=True)

In [ ]:
pdbIDCounts

In [ ]:
dbids.shape

In [ ]:
tmp.shape

In [ ]:

for db,dbc in pdbm.dbCounts.items():
    dbRankData = {rank: dbc[rankCols].sum(axis=1) for rank,rankCols in rankCounts.items()}

In [ ]:
mdbio = ios.get("ClassicalArchives")

In [ ]:
names = mdbio.data.getSummaryNameData()

In [ ]:
rawdata = mdbio.data.getRawComposerFilename(0, 70800)

In [ ]:
rawdata.str

In [ ]:
from ioutils import HTMLIO
hio = HTMLIO()
bsdata = hio.get(io.get(rawdata.str))

In [ ]:
rawdata.str

In [ ]:
bsdata.find("h1").text

In [ ]:
retval = mdbio.prd.rawio.getComposerData(rawdata)

In [ ]:
retval.artist.name

In [ ]:
from utils import MusicDBArtistName
manc = MusicDBArtistName()

In [ ]:
manc.clean(retval.artist.name)

# Jiosaavn

In [ ]:
from gate import IOStore
ios   = IOStore()
mdbio = ios.get("JioSaavn")

In [ ]:
files="""
# /Volumes/Piggy/Discog/artists-jiosaavn/4/m5Sd9I4GRnY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/7/OKd4SFhTD8Q_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/7/b8pscHz-76A_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/10/VCNTgZmUr1c_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/11/VN7USVkKRQo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/12/M7TXqNnYPsA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/12/bwCzD2x32Rk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/12/BTk5GFm5LJg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/12/BWalImcuMWM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/12/DKuMgpba5Qg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/13/QuVtDDvx7nI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/13/XrIJqYqc,zw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/13/PzD6WtmFtFI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/13/xwJvRYgkrG0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/14/A7OD04zu63g_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/14/1t1AmmTr5VU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/14/O1a3J51gWq8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/14/y9qhUexgA9s_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/15/PqciFyWfL8U_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/15/Yz-PA0Uo5FI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/17/y-RMT35Cy1U_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/17/6LCQoroE,t0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/17/pRrngjDBqDw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/17/LIk,SAegJjc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/18/06QxyAvVpB4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/19/AryLS,EwFQ8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/19/C3FZyWBVN0A_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/20/pbvwG-PRzNc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/21/t9ysJ-cCNWY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/21/mVd6dwnk7Dk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/21/YcRkdbYi8Yc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/23/4r3-F646lkk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/23/RwwAdTcWGm8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/24/6T7icLCC2uE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/25/H-1EgXY6P0I_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/25/pNj4flYimBk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/25/MvME6b737bY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/25/hW-I5qJ1gTI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/25/TPgAqKWsJcc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/27/8I8e2sW0UcQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/27/flo0B8MKcpQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/27/wWm7qLQT8l8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/28/X7IBytnK78o_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/28/RMXJZgUbsho_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/28/Pt42bkh9xak_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/28/UGwpEKUZSW4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/28/FPCtG2uvEDA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/29/TNaMFAy7mDg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/30/yaf,eQp2Wcs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/30/7Y0A6jLzlcw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/30/69eGre7Dt2I_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/31/5ItwrzycQPo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/31/FKxk1uxFeD0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/31/s,9ntdn4e8g_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/32/gnIcV0i62zQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/33/2Wuc,xB-LUU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/33/YpNgetHlm6c_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/33/2Ie3ztR2K84_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/34/hn8zrlE47BU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/34/EyWL3SVVYWw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/35/07bF7-wXZlw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/35/,oLCnHktWJk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/35/uytfRp9dKKs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/36/K5CPnDMwUVE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/36/E41rHwqxnvc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/36/JLfecVZ56B4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/36/EfB1xheOhsM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/36/-b0nKQqda4s_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/38/UzREcLMtUBU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/38/P8dtQIu3PVQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/38/p2ZopnWSa84_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/38/CG2qubqX,b4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/40/FYYTX6BP,Eo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/40/LRlhePNM0z0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/41/OV4awU3Ox,8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/41/5jxl4nXzocc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/41/7,EBL-b9bcU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/42/mCN0l1LqzPw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/43/16SbWCzZzao_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/43/9NjFLwMCKtg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/44/KfQ8bRZ0UIg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/45/SgDDb5eX6RU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/45/iF8BkqCTqH0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/45/9lQ6JKKX7yI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/45/XXOZc88UB7c_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/45/,kGyT371bPQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/46/8FJos3ax37U_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/46/FBDzWm8ojoA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/46/6W7JNhbXZaE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/47/iknfllszniM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/47/oaZIMqg0Cl0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/48/8Veku7W188M_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/48/iBpYwFtQRpE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/48/PJTbMExVMh8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/49/aqlopoZLXsM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/49/7QaLYu4R8OI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/50/FiRipxag2zQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/50/b-q,KZHCItk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/50/zdOc4crYu00_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/50/ZNxTI1bQ-yM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/52/,4J8-zZyXVE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/52/56NNhR7g4fg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/52/Vi9wB4RSteM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/52/4pZ2f2l1xq8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/52/4hiHKy9MWV8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/53/-zD87Rb4qcE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/53/35aeeWYrOTY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/53/6W3xq-UMIww_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/53/TK3yylgeymw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/54/lLJeBSwESik_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/54/Ts89qI4E-mo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/54/9,EB4WmLudI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/54/k87qT7vvoPU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/54/mh,RMGJ827g_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/56/J2E4cBqlgpY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/56/fauK4YZMXkE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/57/ht05YtiTBts_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/58/3dIcVnvsaQo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/58/QEZRwVNu2w0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/59/2trg3Z0xljw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/59/5bK7WhdKrxI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/59/Iine5ljEwmw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/59/KHAg3Ww2-pQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/60/XfZGb8-jJjA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/60/RrLwwlfoHpA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/60/P7IdgnwL57c_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/61/yo4keHLYXZQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/61/2igQyp1QPCA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/61/6hv-l7M4amw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/63/qWoOgxru5vQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/63/JH5bNRXxSLY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/64/BgaxIJuhXV4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/64/A1MrJOgms,Y_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/65/lDUUP-jb8tE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/65/LY8K7GCZgh0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/65/GiTO4K84WY4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/66/Bl7bSbDPu7Q_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/66/5hyOWQjDEZo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/66/ePL6LWkdkt4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/66/lZDPeVFm0JM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/66/3np9T0AHZsQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/67/y-frBQdTW68_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/67/U1wqDa9VetY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/67/OEnZZ7k2d5E_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/68/cOhi-v4kNNo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/68/2CzjWL6RJCA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/68/oIhukCMjJNw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/68/Lfq-guDRk0E_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/69/tHl-0fJVeRs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/69/NdXYmSGBz48_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/69/vKmvFW2PC78_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/70/FpN3JUkUwEY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/70/h-Ov6waxaSw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/70/2HnOOZPTJCU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/70/AeHfVLzccwM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/71/7YmxDbch8GE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/71/1LbfPzZgbyY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/71/kw-ATXr252k_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/72/,O5iwgPugBs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/72/nHBaMdTHqCI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/72/yz1RN9iu-8o_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/73/Vt0TXx16f,8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/74/73CTXk,v0jQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/74/vDp3NUOT,hU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/75/Trwat7rfMow_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/75/4ya0E-RtB7k_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/75/8M4qqYmLTdc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/76/VvmU4VnPkts_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/76/IWfuRO2HDgM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/76/yk2Z3vQkXJk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/77/-eFiHrbNXjY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/77/685gEPEEm,w_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/77/ePv6a98WdUI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/80/FjAsWveKW4o_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/80/ekIjZLTfUHA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/80/TXM,LOpPAyc_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/82/zPz-5ylJlL0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/82/oRBk65ofRwQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/82/7ItObd9hj1c_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/84/Rurh5OecoqQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/84/FVYfKBDzsH8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/84/MeCkcUfBwxE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/85/8a-T2kqT3Xo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/85/BWwZ-LLXbLg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/85/ShXMIFs0ZyU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/85/PZztM18o3gE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/86/Qktt2LCDmb0_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/86/x0Z9Glv6Db4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/86/wkx5t1CfUdE_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/86/JJN5UCdTKCY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/87/rqjEn4ityeI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/88/o1-aNOuhK9g_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/88/Lc4CTtipxOI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/88/NidzbDVqo6M_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/88/gI1om24HyXI_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/89/Fu23YAkwOUs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/89/EFextU-GDI8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/90/KZrJRC3eQRA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/90/A8yKhsHxfGs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/91/7qgdRlpfBE4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/91/DXEWeERtrFQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/91/UiJPHgsMQP4_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/92/j1kJn6bs2BY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/92/liNr3EiNqpg_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/92/tZ2MEqtVjuw_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/92/9-o-39OH4OM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/93/k8wSo,mRnWA_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/93/RcYmWKJgJog_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/94/m85ygxPYasY_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/94/zqJBfQm3lEo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/96/Unm5QGbzcWs_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/96/VK79oh2bPYk_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/96/SDAYd-Cd6f8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/96/r3VW2nA1bOo_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/97/MTIdeEdRKwU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/97/aVUEeOlHm6E_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/97/PJ6qIveLVeU_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/97/CtBvRERpwws_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/98/8YGMEuv3bUM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/98/LxUCwY62guQ_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/98/G3p5YiF02SM_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/99/f,3G5vhkbd8_.p
# /Volumes/Piggy/Discog/artists-jiosaavn/99/RfdcbMsNIkY_.p""".split("\n")
from fileutils import FileInfo
files = [FileInfo(ifile.split()[-1]) for ifile in files if len(ifile) > 0]
fsFiles = {ifile.basename: ifile for ifile in files}

In [ ]:
from lib.jiosaavn import RawDBData
from ioutils import FileIO, HTMLIO
hio = HTMLIO()
io = FileIO()
rdb = RawDBData()

for artistID,fInfo in fsFiles.items():
    if fInfo.exists():
        print(fInfo.str)
        retval = rdb.getArtistData(fInfo.str, verbose=False)

In [ ]:
bsdata = hio.get(io.get(fInfo.str))
bsdata

In [ ]:
print(rdb.jsonDataNew)

In [ ]:
x = "1 /\ 2".replace("/\ ", "/\\ ")
x

## Sub

In [ ]:
from master import MasterParams, MusicDBPermDir
mdbpd = MusicDBPermDir()
from base import MusicDBDir, MusicDBData
db = "JioSaavn"
permDBDir = mdbpd.getDBPermPath(db)
permDir = MusicDBDir(permDBDir)
localArtists         = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))

In [ ]:
tmp = localArtists.get()
for artistID in fsFiles.keys():
    del tmp[artistID]
localArtists.save(data=tmp)

In [ ]:
for artistID,fInfo in fsFiles.items():
    if fInfo.exists():
        fInfo.rmFile(debug=True)

In [ ]:
for modVal in range(100):
    mdbio.prd.parse(modVal, force=False, verbose=True)

In [ ]:
from utils import PoolIO
pio = PoolIO("JioSaavn")
#pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

In [ ]:
from lib.jiosaavn import RawDBData
from ioutils import FileIO, HTMLIO
rdb = RawDBData()
hio = HTMLIO()
io = FileIO()
ifile="/Volumes/Piggy/Discog/artists-jiosaavn/86/3UtGDBeB2uY_.p"
bsdata = hio.get(io.get(ifile))
retval = rdb.getArtistData(ifile)

In [ ]:

print(rdb.jsonDataNew)

In [ ]:
bsdata

In [ ]:
retval.show()

In [ ]:
newJSONText

In [ ]:
from apiutils import WebIO
from ioutils import FileIO, HTMLIO
wio = WebIO()
io = FileIO()
hio = HTMLIO()

In [ ]:
url = "https://www.jiosaavn.com/featured/trending_today/I3kvhipIy73uCJW60TJk1Q__"
if False:
    retval = wio.get(url)
    io.save(idata = retval.data, ifile="../sandbox/jiosaavn.p")
    bsdata = hio.get(retval.data)
else:
    from bs4 import BeautifulSoup,element
    bsdata = hio.get(io.get("../sandbox/jiosaavn.p"))

In [ ]:
olElement = bsdata.find("ol")
lis = olElement.findAll("li") if isinstance(olElement,element.Tag) else []
for li in lis:
    print('*'*200)
    print('*'*200)
    print(li)
    h4Element = li.find("h4")
    pElements = li.findAll("p")
    print('-'*200)
    print(h4Element)
    for p in pElements:
        print(p)

In [ ]:
import re
import json
re.search(r'window\.__INITIAL_STATE__ = ({.*});', io.get('../sandbox/jiosaavn.p').decode())
#.group(1)
#data = json.loads(data)

In [ ]:
bsdata

In [ ]:
import sys
from PyQt4.QtGui import QApplication
from PyQt4.QtCore import QUrl
from PyQt4.QtWebKit import QWebPage
import bs4 as bs
import urllib.request

In [ ]:
bsdata

In [ ]:
len(bsdata.findAll("script"))

In [ ]:
#https://www.jiosaavn.com/artist/manan-bhardwaj/PHyqUR9DoKY_

import json
jsData    = [scriptData for scriptData in bsdata.findAll("script") if ((scriptData.get('type') is None))] # and ("window.__INITIAL_DATA__" in scriptData.text))]

In [ ]:
ifile = "/Volumes/Piggy/Discog/artists-jiosaavn/48/hHa1Ha3UMeU_.p"

In [ ]:
x = "\(Femail)".replace("\(F", "(F")

In [ ]:
x

In [ ]:
retval.show()

In [ ]:
artistView = rdb.jData['artistView']
globalView = rdb.jData['global']

In [ ]:
weeklyTop15  = {lang: {'id': item['id'], 'url': item['perma_url']} for lang,item in globalView['globalConfig']['weeklyTop15'].items()}
topArtists   = globalView['globalConfig']['mega_menu']['top_artists']
topPlaylists = globalView['globalConfig']['mega_menu']['top_playlists']
newReleasess = globalView['globalConfig']['mega_menu']['new_releases']

In [ ]:
import json
scriptData = bsdata.find("script", {"type": "application/ld+json"})
try:
    jData = json.loads(scriptData.text)
except:
    jData = {}

In [ ]:
artistData    = artistView['artist']
artistModules = artistView['modules']

In [ ]:
artistModules[0]

In [ ]:
artistData['dob']

In [ ]:
print(newjData)
#jData     = jData.replace("new Date", "").strip() #replace("\n", "").strip()
#jData     = jData[:-1].strip() if jData.endswith(",") else jData.strip()
#jsonData = json.loads(jData)

In [ ]:
help(jData.find)

In [ ]:
newjData.replace("â\x84\x97 ", "")

In [ ]:
bsdata

In [ ]:
val = """{
    "dragdrop": {
        "hasDropped": false,
        "isDragging": false,
        "isValid": false,
        "items": [],
        "target": {}
    },
    "overflow": {
        "isFixed": false,
        "isOpen": false,
        "isPlayer": false,
        "isQueue": false,
        "showDetails": false,
        "current": {},
        "optionsMenu": {},
        "handleOption": null,
        "handleBack": null,
        "stream_event_item": {},
        "x": 0,
        "y": 0
    }}"""
tmp = json.dumps(val)
res = json.loads(tmp)

print(val)
print(tmp)
print(res)

In [ ]:
len(tmp)

# Artist Type

In [ ]:
mdbio = ios.get("Discogs")
for modVal in range(100):
    mdbio.meta.make(modVal, metatype="Link")

In [ ]:
mdbio.sum.make(sumtype="Link")

In [ ]:
from utils import PoolIO
pio = PoolIO("Discogs")
#pio.parse()
pio.meta()
pio.sum()
#pio.search()

In [ ]:
mdbios = ios.get()

In [ ]:
bios = {db: mdbio.data.getSummaryBioData() for db,mdbio in mdbios.items()}
bios = {db: bioData for db,bioData in bios.items() if bioData is not None}
#['Discogs', 'RateYourMusic', 'MetalArchives', 'Deezer', 'MusicBrainz', 'MyMixTapez']

links = {db: mdbio.data.getSummaryLinkData() for db,mdbio in mdbios.items()}
links = {db: linkData for db,linkData in links.items() if linkData is not None}
#['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'MusicBrainz', 'AlbumOfTheYear', 'MyMixTapez']

In [ ]:


genres = {db: mdbio.data.getSummaryGenreData() for db,mdbio in mdbios.items()}
genres = {db: genreData for db,genreData in genres.items() if genreData is not None}
#['Spotify', 'LastFM', 'RateYourMusic', 'MetalArchives', 'AlbumOfTheYear']

metrics = {db: mdbio.data.getSummaryMetricData() for db,mdbio in mdbios.items()}
metrics = {db: metricData for db,metricData in metrics.items() if metricData is not None}
#['Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'MyMixTapez']

names = {db: mdbio.data.getSummaryNameData() for db,mdbio in mdbios.items()}
names = {db: nameData for db,nameData in names.items() if nameData is not None}
#['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']

counts = {db: mdbio.data.getSummaryCountsData() for db,mdbio in mdbios.items()}
counts = {db: countsData for db,countsData in counts.items() if countsData is not None}
#['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'MusicBrainz', 'AlbumOfTheYear', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']

## Artist Type

In [ ]:
#links["Discogs"]["Type"]
#links["MusicBrainz"]["Type"]
#links["RateYourMusic"]["Type"]  ## This could also do work to extract the role people played

In [ ]:
modValData = None #mdbios["Discogs"].data.getModValData(25)

# MyMixTapez

In [ ]:
from apiutils import WebIO
from ioutils import FileIO, HTMLIO
import json
wio = WebIO()
io  = FileIO()
hio = HTMLIO()

In [ ]:
url="https://mymixtapez.com/album/recents"
retval = wio.get(url)
io.save(idata=retval.getData(), ifile="../../sandbox/recent_mymixtapez.p")

url="https://mymixtapez.com/discover/trending-artists"
retval = wio.get(url)
io.save(idata=retval.getData(), ifile="../../sandbox/trending_mymixtapez.p")

In [ ]:
from lib import mymixtapez
mdbio = mymixtapez.MusicDBIO()
ifile = '/Users/tgadfort/Music/Discog/artists-mymixtapez/11/18311.p'
rdb = mymixtapez.RawDBData()
rData = rdb.getArtistData(ifile)
#bsdata = hio.get(io.get(ifile))

In [ ]:
rData.show()

In [ ]:
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))

In [ ]:
jData['artist-page']['success']['entity']
{'id': 18311,
 'name': 'Lil Pump',
 'twitter': 'lilpump',
 'instagram': 'lilpump',
 'bio': 'Lil Pump is one of the top artist hailing from Miami, Fl.  Pump is best known for his 2017 hit single "Gucci Gang" which peaked at number 3 on the Billboard Hot 100 and went triple platinum.  In 2018, he&s;s collaborated with some of the best in the industry, a list that includes Kanye West, DJ Carnage, Skrillex, XXXTentacion, Maluma, and Swae Lee.',
 'subscribers': 1753}

In [ ]:
for item in jData['artist-page']['success']['albums']:
    print(item)
    break

In [ ]:
item

In [ ]:
artists = {}
bsdata = hio.get(io.get("../../sandbox/recent_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["album-recents-page"]['success']['recents']:
    artists.update({artist['id']: artist['name'] for artist in item['artists']['main']})
    
bsdata = hio.get(io.get("../../sandbox/trending_mymixtapez.p"))
jTag = bsdata.find("script", {"type": "application/json"})
jData = json.loads(jTag.text.replace('&q;', '"'))
for item in jData["trending-artists-page"]['success']['items']:
    artists.update({item['id']: item['name']})

In [ ]:
import lib.mymixtapez

In [ ]:
artists

In [ ]:
from lib import mymixtapez
mio   = mymixtapez.MusicDBIO(verbose=False,local=True,mkDirs=True)
webio = mymixtapez.RawWebData(debug=False)

In [ ]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "10:00am")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
trkRows  = bsdata.findAll("trk-row")
trkRows

In [ ]:
for ref in [ref for ref in bsdata.findAll("a", {"class": "tem-item"})]:
    url="https://www.traxsource.com{0}".format(ref.get('href'))
    lnum = ref.get('href').split('/')[-2]
    name = ref.get('href').split('/')[-1]
    savename = "sandbox/traxsource_{0}.p".format(name)
    
    print(url,end=" ... ")
    retval = wio.get(url)
    io.save(idata=retval.data, ifile=savename)
    print(savename)
    wio.sleep(5)

In [ ]:
genres = [ref for ref in bsdata.findAll("a", {"class": "flt"}) if ref.get('href').startswith('/genre/')]
len(genres)

In [ ]:
for ref in genres:
    url = "https://www.traxsource.com{0}/top".format(ref.get('href'))
    name = ref.get('href').split('/')[-1]
    savename = "sandbox/traxsource_{0}.p".format(name)    
    print(url,end=" ... ")
    retval = wio.get(url)
    io.save(idata=retval.data, ifile=savename)
    print(savename)
    wio.sleep(5)

In [ ]:
from fileutils import DirInfo, FileInfo
starter = None
for i,ifile in enumerate(DirInfo("sandbox").glob("traxsource*.p")):
    bsdata  = hio.get(io.get(ifile))
    refs    = [ref for ref in bsdata.findAll("a") if ref.attrs['href'].startswith("/artist/")]
    df      = DataFrame([{"Name": ref.text, "URL": ref.get('href')} for ref in refs])
    if df.shape[0] == 0:
        print(ifile)
    starter = df if starter is None else concat([df,starter]).drop_duplicates()
    #if i % 3 == 0:
    #    print("Found {0: >5} / {1: <5} Unique Artists".format(df.shape[0], starter.shape[0]))
print("Found {0: >5} / {1: <5} Unique Artists".format(df.shape[0], starter.shape[0]))
io.save(idata=starter, ifile="note/traxsource/starter.p")

In [ ]:
sandbox/traxsource_djrae.p

In [ ]:
bsdata = hio.get(io.get("sandbox/traxsource_tracks.p"))
df = DataFrame({ref.get('data-aid'): {"Name": ref.text, "URL": ref.get('href')} for ref in bsdata.findAll('a', {"class": "com-artists"})+bsdata.findAll('a', {"class": "com-remixers"})}).T
io.save(idata=df, ifile="note/traxsource/starter.p")

In [ ]:
trkRows = bsdata.findAll("div", {"class": "trk-row"})
trkCells = trkRows[1].findAll("div", {"class": "trk-cell"})
for v in trkCells:
    print(v)

In [ ]:
headerNames

In [ ]:
trkRows[1]

In [ ]:
aid = MusicDBID()
aid.get(retval["itemListElement"][1]['item']['@id'])

In [ ]:
bsdata

In [ ]:
retval = RawDBData().getArtistData("sandbox/traxsource_djrae.p")

In [ ]:
retval.show()
#retval.media.media["Tracks"][0].get()

In [ ]:
import json5

In [ ]:
https://www.traxsource.com/artist/382021/dj-rae?cn=tracks&ipp=100&ob=r_date&so=desc&page=1

In [ ]:

                    artistdata   = albumdata.findAll("span")[-1]
                    ref = artistdata.find('a')
                    albumartist  = self.makeRawURLInfoData(name=artist.name, url=url.url) if ref is None else self.makeRawLinkData(ref)
                    

                    if mediaData.get(mediaType) is None:
                        mediaData[mediaType] = []
                    mediaData[mediaType].append(amdc)
                    #if self.debug:
                    #    print("Found Album: [{0}/{1}] : {2}  /  {3}".format(len(amc.media[m

In [ ]:
from ioutils import HTMLIO
hio = HTMLIO()
bsdata = hio.get("sandbox/beatport.p")
starter = {ref.attrs['data-artist']: {"Name": ref.text, "URL": ref.attrs['href']} for ref in bsdata.findAll("a") if ref.attrs.get('data-artist') is not None}
print(len(starter))
bsdata = hio.get("sandbox/beatport_releases.p")
starter.update({ref.attrs['data-artist']: {"Name": ref.text, "URL": ref.attrs['href']} for ref in bsdata.findAll("a") if ref.attrs.get('data-artist') is not None})
print(len(starter))
bsdata = hio.get("sandbox/beatport_tracks.p")
starter.update({ref.attrs['data-artist']: {"Name": ref.text, "URL": ref.attrs['href']} for ref in bsdata.findAll("a") if ref.attrs.get('data-artist') is not None})

In [ ]:
io.save(idata=starter, ifile="note/beatport/starter.p")

In [ ]:
tracksURL   = "https://www.beatport.com/artist/john-summit/610028/tracks"
retval = wio.get(tracksURL)
io.save(idata=retval.data, ifile="sandbox/beatport_tracks.p")
wio.sleep(3)
releasesURL = "https://www.beatport.com/artist/john-summit/610028/releases"
retval = wio.get(tracksURL)
io.save(idata=retval.data, ifile="sandbox/beatport_releases.p")

In [ ]:
afroGenreURL="https://www.beatport.com/genre/afro-house/89/releases?page=2&per-page=150"
retval = wio.get(afroGenreURL)
io.save(idata=retval.data, ifile="sandbox/beatport_genrereleases2.p")

In [ ]:
io.save(idata=retval.data, ifile="sandbox/beatport_genrereleases2.p")

In [ ]:
hio = HTMLIO()
bsdata = hio.get("sandbox/beatport_genrereleases.p")

In [ ]:
[ref for ref in bsdata.findAll("a") if ref.attrs.get('data-artist') is not None]

In [ ]:
import json
pjaxDiv = bsdata.find("div", {"id": "pjax-target"})
scriptData = pjaxDiv.find("script", {"type": "application/ld+json"}) if pjaxDiv is not None else ""
jData = json.loads(scriptData.text)

In [ ]:

bucketItems = bsdata.find("ul", {"class": "limited-bucket-items"})
for li in bucketItems.findAll("li", {"class": "bucket-item"}):
    name     = li.attrs['data-ec-name']
    category = li.attrs['data-ec-category']
    brand    = li.attrs['data-ec-brand']
    variant  = li.attrs['data-ec-variant']
    nameID   = li.attrs['data-ec-id']    
    meta     = li.find("div", {"class": "release-meta"})
    print(name,'\t',category,'\t',variant,'\t',brand,'\t',nameID)
    
    p     = meta.find("p", {"class": "release-title"})
    ref   = p.find('a') if p is not None else None
    url   = ref.attrs['href'] if ref is not None else None
    title = ref.text if ref is not None else None
    
    p       = meta.find("p", {"class": "release-artists"})
    artists = p.findAll('a') if p is not None else None
    
    p     = meta.find("p", {"class": "release-label"})
    label = p.find('a') if p is not None else None
    
    print(artists)
    
    
top10Tracks = bsdata.find("ol", {"class": "top-ten-tracks-list"})
for li in bucketItems.findAll("li", {"class": "bucket-item"}):

    
	  <ol class="bucket-items top-ten-tracks-list ec-bucket">
    
      <li class="bucket-item ec-item top-ten-track" data-ec-position="1"
		data-ec-type="product"
		data-ec-name="Drinkee"

In [ ]:
paras[2]

In [ ]:

    if meta is not None:
        titleP   = meta.find("p", {"class" "release-title"})
        titleRef = titleP.find('a') if titleP is not None else None
        print(titleRef)
        #titleURL = titleRef.attrs['href']

        artistsP   = meta.find("p", {"class" "release-artists"})
        artistRefs = artistsP.findAll("a")
    
        print(artistRefs)

In [ ]:
artistRefs = [ref for ref in bsdata.findAll("a") if ref.attrs.get('data-artist') is not None]
#<a href="/artist/vintage-culture/335498" data-artist="335498">Vintage Culture</a>, 
artistRefs

# Summary Mapping

In [ ]:
if False:
    retval = {}
    sumTypes    = ["Link", "Genre", "Metric", "Bio", "Counts"]
    for db,mdbio in mdbios.items():
        print("{0: <25}".format(db),end="")
        for sumType in sumTypes:
            sumData = eval("mdbio.data.getSummary{0}Data".format(sumType))()
            if isinstance(sumData,DataFrame):
                if retval.get(sumType) is None:
                    retval[sumType] = {}
                retval[sumType][db] = sumData
                print("{0: <9}".format(sumType), end="")
            else:
                print("{0: <9}".format(""), end="")
        print("")

In [ ]:
from pandas import Series, isna
def getMapValue(dbIDKey, lookup):
    if isinstance(lookup,Series):
        if isinstance(dbIDKey,str):
            return lookup.get(dbIDKey)
        elif isinstance(dbIDKey,list):
            return [lookup.get(key) for key in dbIDKey]
        elif isna(dbIDKey):
            return None
        else:
            print(dbIDKey)
            raise ValueError("Not sure how to parse {0}".format(type(dbIDKey)))
    else:
        raise ValueError("Not sure how to use lookup {0}".format(type(lookup)))

In [ ]:
summaryData = {}
for sumType in sumTypes:
    ts = Timestat("Getting {0} Data".format(sumType))
    sumTypeData = {db: eval("mdbio.data.getSummary{0}Data".format(sumType))() for db,mdbio in mdbios.items()}
    ts.update()
    sumTypeData = {db: dbSumTypeData for db,dbSumTypeData in sumTypeData.items() if isinstance(dbSumTypeData,DataFrame)}
    ts.update()

    sumTypeDataDF = {}
    for db,dbID in mmeDF.items():
        ts.update()
        dbSumTypeData = sumTypeData.get(db)
        if isinstance(dbSumTypeData,DataFrame):
            for key,value in dbSumTypeData.items():
                gkey = "_".join([db,key])
                sumTypeDataDF[gkey] = dbID.apply(lambda dbIDKey: getMapValue(dbIDKey, value))
    sumTypeDataDF = DataFrame(sumTypeDataDF)
    ts.stop()
    summaryData[sumType] = sumTypeDataDF

In [ ]:
from ioutils import FileIO
from utils import MusicDBPermDir
mdbpd = MusicDBPermDir()
io = FileIO()
for sumType,sumTypeDataDF in summaryData.items():
    ifile = mdbpd.getMatchPermPath().join("summary{0}.p".format(sumType))
    io.save(idata=sumTypeDataDF, ifile=ifile.str)

In [ ]:
from utils import MusicDBPermDir
mdbpd = MusicDBPermDir()
sumType = "Counts"
ifile = mdbpd.getMatchPermPath().join("summary{0}.p".format(sumType))
countsData = io.get(ifile)

In [ ]:
from master import MasterMetas
mms = MasterMetas(verbose=True)
mms.getMedias()

In [ ]:
mmeDF.applymap(lambda x: isinstance(x,int))

# Genre Analysis

## Google

In [ ]:
* Pop music
* Hip hop music
* Rock music
* Rhythm and blues
* Soul music
* Reggae
* Country
* Funk
* Folk music
* Middle Eastern music
* Jazz
* Disco
* Classical music
* Electronic music
* Music of Latin America
* Blues
* Music for children
* New-age music
* Vocal music
* Music of Africa
* Christian music
* Music of Asia
* Ska
* Traditional music
* Independent music

In [ ]:
Art (classical)
Avant-garde and experimental


In [ ]:
rymGenres="""Ambient
Blues
Classical Music
Comedy
Country
Dance
Electronic
Experimental
Field Recordings
Folk
Hip Hop
Industrial Music
Jazz
Metal
Musical Theatre and Entertainment
New Age
Pop
Psychedelia
Punk
R&B
Regional Music
Rock
Singer-Songwriter
Ska
Sounds and Effects
Spoken Word""".split("\n")

In [ ]:
mdbio = ios.get("RateYourMusic")
genreData = mdbio.data.getSummaryGenreData()

In [ ]:
from collections import Counter
for idx,genre in genreData[genreData["Genre"].notna()].iterrows()

In [ ]:
from apiutils import WebIO
from ioutils import HTMLIO, FileIO
wio = WebIO()
retval = wio.get("https://www.chosic.com/list-of-music-genres/")
data = retval.getData()

hio = HTMLIO()
io = FileIO()
bsdata = hio.get(data)
io.save(idata=data, ifile="genre.p")

In [ ]:
bsdata = hio.get("genre.p")

uls = [ul for ul in bsdata.findAll("ul") if ul.find("li", {"class": "genre-term-basic"}) is not None]
baseUL = uls[0] if len(uls) == 1 else None
if baseUL is not None:
    genres = baseUL.findAll("li", {"class": "genre-term-basic"})
    subULs = baseUL.findAll("ul", {"class": "ul-inside"})
    genreData = {genre.text: {li.text: li.find('a').attrs['href'] for li in subUL} for genre,subUL in dict(zip(genres, subULs)).items()}
    
    def fixGenre(genre):
        genre = genre.replace(" (also called Contemporary folk - wikipedia)", "")
        genre = genre.replace("(wikipedia)", "")
        genre = genre.replace(" (Electronic Dance Music)", "")
        genre = genre if genre == genre.upper() else genre.title()
        return genre
    
    genreData = {fixGenre(genre): [fixGenre(subGenre) for subGenre in subGenres.keys()] for genre,subGenres in genreData.items()}

In [ ]:
from timeutils import Timestat
from listUtils import getFlatList
from pandas import isna

caps = {"Edm": "EDM", "Opm", "OPM", "Uk": "UK", "Emo": "EMO"}

def fixGenre(genre):
    genre = genre if genre == genre.upper() else genre.title()
    genre = genre.upper() if genre in ["Edm", "Opm", "Emo"] else genre
    return genre

def fixLists(x):
    if x is None:
        return None
    if isinstance(x,list):
        retval = []
        for item in x:
            if isinstance(item,str):
                retval.append(fixGenre(item))
            elif isinstance(item,list):
                for subitem in item:
                    if isinstance(subitem,str):
                        retval.append(fixGenre(subitem))
            elif item is None or isna(item):
                continue
            else:
                raise ValueError("What is this?: {0}".format(item))
        return retval
    elif isna(x):
        return None
    else:
        raise ValueError("What is this?: {0}".format(x))

def summaryGenre(mdbios, mmeDF, sumType="Genre"):    
    ts = Timestat("Getting {0} Data".format(sumType))
    sumTypeData = {db: eval("mdbio.data.getSummary{0}Data()".format(sumType)) for db,mdbio in mdbios.items()}
    sumTypeData = {db: dbSumTypeData for db,dbSumTypeData in sumTypeData.items() if isinstance(dbSumTypeData,DataFrame)}
    N = len(sumTypeData)
    ts.stop()
    
    sumTypeDataDF = {}
    ts = Timestat("Matching {0} Data".format(sumType))
    for n,(db,dbSumTypeData) in enumerate(sumTypeData.items()):
        dbID = mmeDF[db]
        ts.update(n=n+1,N=N)
        if isinstance(sumTypeData.get(db),DataFrame):
            for key,value in sumTypeData[db].items():
                gKey = "{0}_{1}".format(db,key)
                if db in ["Spotify"]:
                    sumTypeDataDF[gKey] = dbID.map(lambda dbIDKey: [value.get(y,[]) for y in dbIDKey] if isinstance(dbIDKey,list) else value.get(dbIDKey,[]))
                else:
                    sumTypeDataDF[gKey] = dbID.map(value)

    ts.stop()
    retval = DataFrame({gKey: gKeyValue.apply(fixLists) for gKey,gKeyValue in sumTypeDataDF.items()})
    return retval

In [ ]:
sumTypeDataDF = summaryGenre(mdbios, mmeDF)

In [ ]:
sumTypeDataDF

In [ ]:
from collections import Counter
genres = {}
for key,value in sumTypeDataDF.iteritems():
    genres[key] = Counter()
    for idx,genre in value[value.notna()].iteritems():
        if isinstance(genre,list):
            for item in genre:
                genres[key][item] += 1
                
genres = {key: Series(value).sort_values(ascending=False) for key,value in genres.items()}
for key,value in genres.items():
    value.name = key


In [ ]:
genreHierachy = {subGenre: genre for genre,subGenres in io.get("GenreHierachy.yaml").items() for subGenre in subGenres}
df = DataFrame(genres["Spotify_Genre"])
df["Hierarchy"] = df["Spotify_Genre"].index.map(genreHierachy)

In [ ]:
genreHierachy = io.get("GenreHierachy.yaml")
allSubGenres = {}
primeGenres = genreHierachy.keys()
for genre,subGenres in genreHierachy.items():
    for subGenre in subGenres:
        if allSubGenres.get(subGenre) is None: # and subGenre not in primeGenres.:
            allSubGenres[subGenre] = genre
        else:
            print('===>',subGenre,':',genre,':',allSubGenres.get(subGenre))
            #raise ValueError("Multiple {0}".format(subGenre))


In [ ]:
saveData = {}
for subGenre, genre in allSubGenres.items():
    if saveData.get(genre) is None:
        saveData[genre] = []
    saveData[genre].append(subGenre)
saveData = {genre: sorted(subGenres) for genre,subGenres in saveData.items()}
io.save(idata=saveData, ifile="GenreHierachy.yaml")

In [ ]:
for idx in df[(df["Hierarchy"].isna()) & (df.index.str.contains(" Punk"))].index:
    print("- {0}".format(idx))

In [ ]:
df[(df["Hierarchy"].isna())].head(20)

# Base Artist Class

In [ ]:
from pandas import isna

class MasterKey(str):
    def __init__(self, key):
        assert isinstance(key,(str,int,MasterKey)), "Key must be a str, int, or self. You gave a [{0}]".format(type(key))
        self = str(key)
        
mk = MasterKey(49)
mk == '49'

In [ ]:
from pandas import Timestamp

class MasterTimestamp:
    def __init__(self):
        now = Timestamp.today().round('min')
        self.created  = now
        self.modified = now
        
    def update(self):
        self.modified = Timestamp.today().round('min')

In [ ]:
from masterDBGate import masterDBGate
from pandas import Series, DataFrame
from uuid import uuid4

class masterArtistDB:
    def __init__(self):
        self.db = Series(dtype = 'object')
        
    def add(self, ma):
        row = Series({ma.id: ma})
        self.db = self.db.append(row, verify_integrity=True)
        
    def N(self):
        return len(self.db)
        
    #def getData(self):
        
        

class MasterArtist:
    def __init__(self, name, dbids, **kwargs):
        ### DB Entry Information
        self.ts = MasterTimestamp()
        self.id = uuid4()
        
        mp = MasterParams()

        
        assert isinstance(self.name,str), "'name' must be a string"
        self.name = name
        
        assert isinstance(self.dbids,dict), "'name' must be a dict"
        self.dbids = {}
        for db,dbid in dbids.items():
            assert mp.isValid(db), "DB [{0}] is not valid".format(db)
            self.dbids[db] = MasterKey(dbid)
        
        
    def setName(self, name):
        assert isinstance(name,str), "'name' must be a string"
        self.name = name
        self.ts.update()
                
    def setdbid(self, db, dbID):
        assert mp.isValid(db), "DB [{0}] is not valid".format(db)
        if isinstance(self.dbIDs.get(db), masterArtistKey):
            self.dbIDs[db] = masterArtistKey(dbID)
            
        
    def get()

# Parallel Match Studies

In [ ]:
matchedIDs      = mmeDF[rym.db][mmeDF[rym.db].notna()]
unmatchedDBData = basicData[~basicData.index.isin(matchedIDs)].sort_values(by="NumAlbums", ascending=False)
print("There are {0} Artists In {1} DB".format(basicData.shape[0], rym.db))
print("There are {0} Artists Matched In PanDB".format(matchedIDs.shape[0]))
print("    ====> {0} UnMatched In {1} DB".format(unmatchedDBData.shape[0], rym.db))

In [ ]:
mdbSearchData = unmatchedDBData[unmatchedDBData["NumAlbums"] >= 15].join(rym.data.getSearchAlbumMediaData()).join(rym.data.getSearchSingleEPMediaData())
mdbSearchData.shape

In [ ]:
mdbSearchData = unmatchedDBData[unmatchedDBData["NumAlbums"] >= 15].join(rym.data.getSearchAlbumMediaData()).join(rym.data.getSearchSingleEPMediaData())
mdbSearchData.shape

compareData = DataFrame(spot.data.getSearchNameData()).join(spot.data.getSummaryNumAlbumsData())
compareData = compareData.sort_values(by="NumAlbums", ascending=False)

from utils import poolDataFrame

In [ ]:
help(pool.map)

In [ ]:
from numpy import array_split
from multiprocessing import Pool
    #df_split = array_split(df, n_cores)
pool     = Pool(2)
df_split = array_split(mdbSearchData["Name"], 2)
retval   = pool.map(func=cd.getNearestNames, iterable=df_split)

In [ ]:
mdbSearchData["Name"]

In [ ]:
from numpy import vectorize

class CompareData:
    def __init__(self, data, cutoff=0.8):
        self.data = data
        self.cutoff = cutoff
        self.vCompare = vectorize(self.compare)
        
    def getLevenshtein(self, x1, x2):
        try:
            return Levenshtein.ratio(x1, x2)
        except:
            print("ERROR With Levenshtein.ratio({0}, {1})".format(x1,x2))
            return 0.0        
        
    def compare(self, artistName):
        return self.data["Name"].apply(self.getLevenshtein, x2=artistName)
    
    def getNearestNames(self, artistName):
        print(artistName)
        retval = self.data["Name"].apply(self.getLevenshtein, x2=artistName)
        return retval
        #retval = Series(self.vCompare(artistName), index=self.data.index)
        print(artistName,'\t',retval)
        return self.data.index
        #return self.data.index[self.vCompare(artistName) > self.cutoff] #mdbSearchData["Name"])
        return ["Some Index"]

In [ ]:
mdbSearchData["Name"].loc['1004134']

In [ ]:
%%time
ddata = dd.from_pandas(mdbSearchData["Name"], npartitions=2).map_partitions(lambda df: df.apply(cd.compare)).compute(scheduler='processes')   # Wall time: 35.9 s
#retval = poolDataFrame(df=mdbSearchData["Name"], func=cd.getNearestNames)

In [ ]:
cd.data.head()

In [ ]:
ddata

In [ ]:
import Levenshtein
def getLevenshtein(x1, x2):
    try:
        return Levenshtein.ratio(x1, x2)
    except:
        print("ERROR With Levenshtein.ratio({0}, {1})".format(x1,x2))
        return 0.0
    
def compare(artistName):
    return compareData["Name"].apply(getLevenshtein, x2=artistName)

def compareMax(artistName):
    return compareData["Name"][abs(compareData["Name"].str.len() - len(artistName)) < 5].apply(getLevenshtein, x2=artistName)

from numpy import vectorize
vLev = vectorize(getLevenshtein)
vCompare = vectorize(compare)
vCompareMax = vectorize(compareMax)

In [ ]:
%%time
#def v
#Levenshtein.ratio(x1, x2)


In [ ]:
help(compareData["Name"].map)

In [ ]:
%%time
#_ = [vLev(compareData["Name"], name) for name in mdbSearchData["Name"]]
mdbSearchData["Name"].swifter.apply(lambda x: compareData["Name"].swifter.apply(vLev, x2=x))

In [ ]:
vLev()
vectorize(vLev, compareData["Name"]

In [ ]:
def 

In [ ]:
%%time
#mdbSearchData["Name"].apply(compare)           # Wall time: 43.2 s for 435 rows × 101432 columns
#mdbSearchData["Name"].swifter.apply(compare)   # Wall time: 1min 27s 435 rows × 101432 columns
#ddata = dd.from_pandas(mdbSearchData["Name"], npartitions=10).map_partitions(lambda df: df.apply(vCompare)).compute(scheduler='processes')   # Wall time: 49.6 s

#ddata = dd.from_pandas(mdbSearchData["Name"], npartitions=10).map_partitions(lambda df: df.apply(compare)).compute(scheduler='processes')   # Wall time: 35.9 s
#ddata = dd.from_pandas(mdbSearchData["Name"], npartitions=10).map_partitions(lambda df: df.apply(compareMax)).compute(scheduler='processes')   # Wall time: 35.9 s

#mdbSearchData["Name"].swifter.apply(vCompare)
#                                    (compareData["Name"][abs(compareData["Name"].str.len() - len(x)) < 2], x))   # Wall time: 32.5 s
#compareData["Name"].swifter.apply(lambda x: vLev(mdbSearchData["Name"], x))

#mdbSearchData["Name"].swifter.apply(vCompare)  # Wall time: 1min 38s
#mdbSearchData["Name"].swifter.apply(compare)  # Wall time: 1min 24s
dd.from_pandas(mdbSearchData["Name"], npartitions=10).map_partitions(lambda df: df.apply(compare)).compute(scheduler='processes')   # Wall time: 35.9 s